# Solution 01 — Policy Conflict Analysis

Reference implementation for Assessment 01. Achievable in ~45 minutes with plain Python + `call_llm`.


## Problem Recap

Given a multi-section corporate policy, produce a structured report of **conflicts**, **ambiguities**, and **exception risks**, plus an analyst summary.


## Architectural Decisions

We use a **decompose → analyze → refine** pipeline:

1. **Segment** the policy into labeled sections (one call) so later steps cite precise clause IDs.
2. **Conflict & loophole pass** over the structured sections (one call) focused on contradictions and abuse paths.
3. **Ambiguity pass** (one call) focused on vague language — separate from hard conflicts so the model does not conflate the two.
4. **Validate & synthesize** (one call): merge findings, enforce schema, write the analyst summary, drop duplicates.

Why not one giant call? Single-shot analysis on long policies tends to miss cross-section interactions and mixes severity. Separating structure from critique improves citation quality within a short interview.


In [ ]:
import json
import os
import re
from typing import Any, Optional

# ---------------------------------------------------------------------------
# LLM configuration
# ---------------------------------------------------------------------------
# Set one of the following before running (or export in your shell):
#
#   export OPENAI_API_KEY="sk-..."
#   # optional: export OPENAI_MODEL="gpt-4o-mini"
#
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   # optional: export ANTHROPIC_MODEL="claude-3-5-haiku-latest"
#
# If neither key is set, call_llm runs in MOCK mode so the notebook still
# executes for smoke-testing. Mock responses are placeholders — they are not
# a substitute for a real model during practice.
# ---------------------------------------------------------------------------

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")

USE_MOCK = not (OPENAI_API_KEY or ANTHROPIC_API_KEY)
if USE_MOCK:
    print("⚠️  No API key found. call_llm is in MOCK mode.")
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"✓ Using {provider} API")


def _mock_response(prompt: str, system_prompt: str = "") -> str:
    """Lightweight deterministic mock for offline smoke tests."""
    combined = f"{system_prompt}\n{prompt}".lower()
    # Return minimal JSON-ish or text so downstream parsing can be exercised.
    if "json" in combined:
        return json.dumps({"mock": True, "note": "Replace with real API key for useful output", "items": []})
    return "[MOCK RESPONSE] Configure OPENAI_API_KEY or ANTHROPIC_API_KEY for real model output."


def call_llm(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.2,
    max_tokens: int = 2000,
) -> str:
    """Call a chat LLM. Credentials come from environment variables only."""
    if USE_MOCK:
        return _mock_response(prompt, system_prompt)

    try:
        if OPENAI_API_KEY:
            from openai import OpenAI

            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        from anthropic import Anthropic

        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        ).strip()
    except Exception as e:
        return f"[API ERROR] {type(e).__name__}: {e}"


def extract_json(text: str) -> Any:
    """Best-effort parse of a JSON object/array from model output."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except json.JSONDecodeError:
            pass
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Could not parse JSON from model output:\n{text[:500]}")


# Smoke test
print("call_llm ready.")
print(call_llm("Reply with exactly: OK", system_prompt="Be terse.")[:80])


In [ ]:
POLICY_TEXT = """
CORPORATE TRAVEL & EXPENSE POLICY (v3.2) — Acme Corp

SECTION 1 — Booking Authority
1.1 Employees may book domestic flights under $800 without manager approval.
1.2 International flights require VP approval regardless of cost.
1.3 The Travel Desk may override booking rules for executive assistants supporting C-level staff.

SECTION 2 — Lodging
2.1 Hotel stays are capped at $250/night in Tier-1 cities and $175/night elsewhere.
2.2 Employees attending company-sponsored conferences may book the official conference hotel
    even if the nightly rate exceeds the standard cap, with no additional approval required.
2.3 Suite upgrades are prohibited unless medically necessary and approved by HR.

SECTION 3 — Meals & Incidentals
3.1 Daily meal allowance is $75 domestic / $100 international.
3.2 Client entertainment meals may exceed the daily allowance if a receipt and client names
    are attached; no hard dollar cap applies.
3.3 Alcohol is reimbursable only when accompanying a client meal.

SECTION 4 — Exceptions & Escalation
4.1 Any expense over $2,000 requires Finance Controller pre-approval.
4.2 The Travel Desk exception process (Form T-9) can approve any single expense up to $5,000
    without Finance Controller involvement if submitted within 48 hours of booking.
4.3 Retroactive exceptions after travel are not permitted.

SECTION 5 — Card & Reimbursement
5.1 All travel must be charged to the corporate card when available.
5.2 Personal card use is allowed only when the corporate card is declined; reimbursement
    then requires manager sign-off within 5 business days.
5.3 Employees in Sales Ops are exempt from the corporate-card requirement and may use
    personal cards for all travel expenses under $1,500.
"""


## Why Each Model Call Is Necessary

| Call | Purpose |
|------|---------|
| Segment | Normalize messy text into citeable units; reduces hallucinated section IDs later |
| Conflicts/risks | Force attention onto pairwise/cross-rule contradictions |
| Ambiguities | Separate soft-language issues from hard conflicts |
| Merge/validate | Deduplicate, fill schema gaps, produce executive summary |


In [ ]:
def analyze_policy_conflicts(policy_text: str) -> dict:
    """Reference compound system: segment → conflict/risk → ambiguity → merge."""
    empty = {
        "conflicts": [],
        "ambiguities": [],
        "exception_risks": [],
        "analyst_summary": "Insufficient policy content to analyze.",
    }
    if not policy_text or len(policy_text.strip()) < 40:
        return empty

    # --- Call 1: segment into citeable units ---
    segment_raw = call_llm(
        prompt=(
            "Break the following policy into a JSON array of objects with keys "
            "`id` (section number like '4.2' if present, else invent 'X.N'), "
            "`title` (short), and `text` (verbatim clause text). "
            "Return ONLY JSON.\n\nPOLICY:\n" + policy_text
        ),
        system_prompt="You extract structured sections from policy documents. Output JSON only.",
        temperature=0.0,
    )
    try:
        sections = extract_json(segment_raw)
        if isinstance(sections, dict):
            sections = sections.get("sections", sections.get("items", []))
    except ValueError:
        sections = [{"id": "ALL", "title": "full", "text": policy_text}]

    sections_blob = json.dumps(sections, indent=2)[:12000]

    # --- Call 2: conflicts + exception risks ---
    conflict_raw = call_llm(
        prompt=(
            "Given policy sections as JSON, identify (1) hard conflicts where following "
            "one clause violates another, and (2) exception pathways that create abuse risk.\n"
            "Return JSON: {\"conflicts\":[...], \"exception_risks\":[...]} using this schema:\n"
            "conflicts: id, summary, sections_involved, severity (high|medium|low), "
            "explanation, example_scenario\n"
            "exception_risks: id, summary, sections_involved, abuse_potential "
            "(high|medium|low), explanation\n"
            "Cite section ids from the input. Return ONLY JSON.\n\nSECTIONS:\n"
            + sections_blob
        ),
        system_prompt="You are a compliance analyst hunting contradictions and loopholes.",
        temperature=0.1,
    )
    try:
        conflict_part = extract_json(conflict_raw)
    except ValueError:
        conflict_part = {"conflicts": [], "exception_risks": []}

    # --- Call 3: ambiguities (separate focus) ---
    amb_raw = call_llm(
        prompt=(
            "Given the same policy sections, list ambiguities and vague language that cause "
            "inconsistent enforcement (not hard logical conflicts).\n"
            "Return JSON: {\"ambiguities\":[{id, summary, sections_involved, explanation, "
            "clarifying_question}]}\nReturn ONLY JSON.\n\nSECTIONS:\n" + sections_blob
        ),
        system_prompt="You flag ambiguous policy language for Legal clarification.",
        temperature=0.1,
    )
    try:
        amb_part = extract_json(amb_raw)
    except ValueError:
        amb_part = {"ambiguities": []}

    draft = {
        "conflicts": conflict_part.get("conflicts", []) if isinstance(conflict_part, dict) else [],
        "exception_risks": conflict_part.get("exception_risks", []) if isinstance(conflict_part, dict) else [],
        "ambiguities": amb_part.get("ambiguities", []) if isinstance(amb_part, dict) else [],
    }

    # --- Call 4: refine, dedupe, summarize ---
    final_raw = call_llm(
        prompt=(
            "Merge and validate the draft findings below. Deduplicate overlapping items, "
            "ensure IDs are unique (C1.., A1.., R1..), severity fields are valid, and "
            "write analyst_summary (2-4 sentences).\n"
            "Return ONLY JSON with keys: conflicts, ambiguities, exception_risks, analyst_summary.\n\n"
            f"DRAFT:\n{json.dumps(draft, indent=2)[:10000]}\n\n"
            f"ORIGINAL POLICY (excerpt):\n{policy_text[:3000]}"
        ),
        system_prompt="You produce clean compliance review JSON. No markdown.",
        temperature=0.1,
    )
    try:
        result = extract_json(final_raw)
    except ValueError:
        result = {**draft, "analyst_summary": "Draft findings returned without successful merge pass."}

    for key in ("conflicts", "ambiguities", "exception_risks"):
        result.setdefault(key, [])
        if not isinstance(result[key], list):
            result[key] = []
    result.setdefault("analyst_summary", "")
    return result


## Example Execution


In [ ]:
result = analyze_policy_conflicts(POLICY_TEXT)
print(json.dumps(result, indent=2)[:4000])
print("\n--- counts ---")
print({k: len(result[k]) for k in ("conflicts", "ambiguities", "exception_risks")})


## Alternative Architectures

- **Single-shot JSON**: Faster/cheaper; often misses cross-section conflicts on longer policies.
- **Pairwise section comparison**: More thorough for N sections but O(N²) calls — too slow for 45 minutes and for production without filtering.
- **Retrieve-then-critique**: Embed clauses and retrieve likely-related pairs before LLM critique — better at scale, more setup than an interview allows.

## Accuracy / Latency / Cost

Four calls ≈ 4× latency vs one-shot. Worth it for citation quality. Temperature kept low for determinism. Mock mode proves wiring only.

## Common Mistakes

- Asking for "summary of the policy" instead of conflicts.
- No section IDs → analyst cannot verify.
- Stuffing conflicts and ambiguities into one muddy list.
- Ignoring exception-stacking loopholes (the real enterprise pain).

## Beyond 45 Minutes

- Confidence scores per finding; human-in-the-loop review UI.
- Diff mode across policy versions.
- Regression suite of synthetic conflict fixtures.
- Parallelize conflict and ambiguity calls.
